# 04b. Modèle de Fondation Zero-Shot : Chronos-2 (Amazon)

**Projet :** Prévision de l'or (XAU/USD) — *Advanced Time Series* (ENSAI, 3A)

### Objectifs de ce notebook :
1. Tester un **Time Series Foundation Model (TSFM)** récent (**Chronos-2**, 120M paramètres) en **zero-shot** (sans réentraînement spécifique sur l'or).
2. Comparer deux configurations :
   - **Chronos-2 Univarié** : uniquement l'historique des log-rendements de l'or.
   - **Chronos-2 avec Covariables passées** : ajout de `DXY`, `VIX`, `TNX_10Y`, `OIL`, `SP500`.
3. Effectuer une **validation walk-forward ($h=1$)** sur une fenêtre récente hors échantillon.
4. Vérifier empiriquement le constat de la littérature financière récente (nov. 2026) : les TSFM généralistes souffrent-ils de **forecast collapse** sur les rendements bruités, et s'en sortent-ils mieux sur la **volatilité** ($|r_t|$) ?

In [ ]:
import os
# Désactive XetHub si le réseau bloque cas-server.xethub.hf.co
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from chronos import BaseChronosPipeline

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (13, 5)

DATA_DIR = Path("../data")
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch {torch.__version__} | CPU threads: {torch.get_num_threads()}")

## 1. Préparation des données & Régularisation calendaire

Chronos-2 (`predict_df`) infère la fréquence temporelle de la série. Les jours fériés boursiers créant des trous irréguliers, on réindexe sur les **jours ouvrés (`freq='B'`)** avec forward-fill des prix de clôture avant de calculer les log-rendements.

In [ ]:
df_gold = pd.read_csv(DATA_DIR / "gold_daily.csv", parse_dates=["Date"]).set_index("Date")
df_cov = pd.read_csv(DATA_DIR / "covariates_daily.csv", parse_dates=["Date"]).set_index("Date")

# Fusion et régularisation en jours ouvrés
df_raw = df_gold[["Close"]].join(df_cov, how="left").asfreq("B").ffill()

# Construction des séries stationnaires
df = pd.DataFrame(index=df_raw.index)
df["Close"] = df_raw["Close"]
df["return"] = np.log(df_raw["Close"]).diff()
df["abs_return"] = np.abs(df["return"])

# Variations des covariables macroéconomiques
df["DXY_ret"] = np.log(df_raw["DXY"]).diff()
df["VIX"] = df_raw["VIX"]
df["TNX_diff"] = df_raw["TNX_10Y"].diff()
df["OIL_ret"] = np.log(df_raw["OIL"].clip(lower=0.01)).diff()
df["SP500_ret"] = np.log(df_raw["SP500"]).diff()

df = df.dropna().reset_index()
df["item_id"] = "XAUUSD"

print(f"Observations (jours ouvrés) : {len(df)} | {df['Date'].min().date()} -> {df['Date'].max().date()}")
df.tail(3)

## 2. Chargement de Chronos-2 (`amazon/chronos-2`)

In [ ]:
# Sur MacBook Air, device_map="cpu" est stable et rapide pour l'inférence zero-shot
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"
)
print("Modèle Chronos-2 chargé avec succès.")

## 3. Aperçu Zero-Shot sur un horizon de 10 jours (Univarié vs Covariables)

On garde les **10 derniers jours ouvrés** comme test visuel, et on fournit les 512 jours précédents en contexte.

In [ ]:
H = 10
CONTEXT = 512

hist_uni = df.iloc[-(H + CONTEXT):-H][["item_id", "Date", "return"]].copy()
cov_cols = ["DXY_ret", "VIX", "TNX_diff", "OIL_ret", "SP500_ret"]
hist_multi = df.iloc[-(H + CONTEXT):-H][["item_id", "Date", "return"] + cov_cols].copy()
actual_h = df.iloc[-H:][["Date", "return"]].copy()

pred_uni = pipeline.predict_df(
    hist_uni, prediction_length=H, quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id", timestamp_column="Date", target="return", freq="B"
)

pred_multi = pipeline.predict_df(
    hist_multi, prediction_length=H, quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id", timestamp_column="Date", target="return", freq="B"
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df.iloc[-40:-H+1]["Date"], df.iloc[-40:-H+1]["return"], color="gray", alpha=0.6, label="Historique récent")
ax.plot(actual_h["Date"], actual_h["return"], color="black", marker="o", lw=1.5, label="Réalisé")
ax.plot(pred_uni["Date"], pred_uni["0.5"], color="#1f77b4", lw=2, label="Chronos-2 Univarié (médiane)")
ax.fill_between(pred_uni["Date"], pred_uni["0.1"], pred_uni["0.9"], color="#1f77b4", alpha=0.15, label="Intervalle 10%-90%")
ax.plot(pred_multi["Date"], pred_multi["0.5"], color="#d62728", linestyle="--", lw=2, label="Chronos-2 + Covariables (médiane)")
ax.axhline(0, color="black", lw=0.8, linestyle=":")
ax.set_title("Chronos-2 Zero-Shot sur les log-rendements de l'or (h=10 jours)", fontweight="bold")
ax.set_ylabel("Log-rendement")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "04b_chronos_horizon10.png", dpi=300)
plt.show()

## 4. Validation Walk-Forward ($h=1$ jour) sur les 60 derniers jours ouvrés

Pour comparer équitablement Chronos-2 au **Benchmark Naïf ($\hat{r}_t = 0$)** et aux modèles ARIMA/ML dans `05_evaluation.ipynb`, on évalue la prévision à **1 pas en avant ($h=1$)** sur les **60 derniers jours**.

*Astuce de performance :* On regroupe les 60 fenêtres glissantes en un seul DataFrame avec des `item_id` distincts (`win_0` ... `win_59`) pour que Chronos-2 les traite en **un seul batch** !

In [ ]:
N_TEST = 60
CTX_LEN = 252  # 1 an de bourse de contexte par fenêtre

windows_uni = []
windows_multi = []
truth_rows = []

start_idx = len(df) - N_TEST
for i in range(N_TEST):
    t = start_idx + i
    win_id = f"win_{i:03d}"
    
    sub_multi = df.iloc[t - CTX_LEN : t][["Date", "return"] + cov_cols].copy()
    sub_multi["item_id"] = win_id
    windows_multi.append(sub_multi)
    
    sub_uni = sub_multi[["item_id", "Date", "return"]].copy()
    windows_uni.append(sub_uni)
    
    truth_rows.append({
        "item_id": win_id,
        "Date": df.iloc[t]["Date"],
        "y_true": df.iloc[t]["return"],
        "abs_true": df.iloc[t]["abs_return"]
    })

batch_uni = pd.concat(windows_uni, ignore_index=True)
batch_multi = pd.concat(windows_multi, ignore_index=True)
df_truth = pd.DataFrame(truth_rows)

print(f"Lancement du walk-forward batch ({N_TEST} fenêtres)...")
wf_uni = pipeline.predict_df(
    batch_uni, prediction_length=1, quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id", timestamp_column="Date", target="return", freq="B", batch_size=32
)
wf_multi = pipeline.predict_df(
    batch_multi, prediction_length=1, quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id", timestamp_column="Date", target="return", freq="B", batch_size=32
)

# Consolidation des résultats
res = df_truth.copy()
res["pred_naive"] = 0.0
res["pred_chronos_uni"] = wf_uni.sort_values("item_id")["0.5"].values
res["pred_chronos_multi"] = wf_multi.sort_values("item_id")["0.5"].values
res["q10_uni"] = wf_uni.sort_values("item_id")["0.1"].values
res["q90_uni"] = wf_uni.sort_values("item_id")["0.9"].values

res.head()

In [ ]:
def eval_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    # Précision directionnelle (ignorant les prédictions exactement nulles pour le naïf)
    hit = np.mean(np.sign(y_true) == np.sign(y_pred)) if np.any(y_pred != 0) else 0.50
    std_pred = np.std(y_pred)
    return {"Modèle": name, "MAE": mae, "RMSE": rmse, "Hit Ratio": hit, "Std(pred)": std_pred}

metrics_df = pd.DataFrame([
    eval_metrics(res["y_true"], res["pred_naive"], "Naïf (r_t = 0)"),
    eval_metrics(res["y_true"], res["pred_chronos_uni"], "Chronos-2 Univarié"),
    eval_metrics(res["y_true"], res["pred_chronos_multi"], "Chronos-2 + Covariables"),
])

print(metrics_df.to_string(index=False))

# Sauvegarde pour 05_evaluation.ipynb
res.to_csv(DATA_DIR / "preds_chronos.csv", index=False)
print("\nPrévisions sauvegardées dans data/preds_chronos.csv")

## 5. Test bonus : Chronos-2 sur l'amplitude / volatilité ($|r_t|$)

Comme le souligne la littérature récente, les rendements bruts sont très bruités (proches d'une marche aléatoire), mais **l'amplitude $|r_t|$** (proxy de volatilité) est fortement structurée. Voyons si Chronos-2 capte mieux cette dynamique !

In [ ]:
windows_vol = []
for i in range(N_TEST):
    t = start_idx + i
    sub_v = df.iloc[t - CTX_LEN : t][["Date", "abs_return"]].copy()
    sub_v["item_id"] = f"win_{i:03d}"
    windows_vol.append(sub_v)

batch_vol = pd.concat(windows_vol, ignore_index=True)
wf_vol = pipeline.predict_df(
    batch_vol, prediction_length=1, quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id", timestamp_column="Date", target="abs_return", freq="B", batch_size=32
)

vol_pred = wf_vol.sort_values("item_id")["0.5"].values
vol_naive = df["abs_return"].iloc[start_idx - 1 : start_idx + N_TEST - 1].values

print(f"Volatilité |r_t| - RMSE Naïf (t-1) : {root_mean_squared_error(res['abs_true'], vol_naive):.6f}")
print(f"Volatilité |r_t| - RMSE Chronos-2  : {root_mean_squared_error(res['abs_true'], vol_pred):.6f}")

plt.figure(figsize=(13, 4))
plt.plot(res["Date"], res["abs_true"], label="Volatilité réalisée |r_t|", color="black", alpha=0.7)
plt.plot(res["Date"], vol_pred, label="Chronos-2 Zero-Shot |r_t|", color="#2ca02c", lw=2)
plt.title("Chronos-2 Zero-Shot sur la volatilité (|r_t|)", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "04b_chronos_volatility.png", dpi=300)
plt.show()

## 6. Expérience avancée : Fine-Tuning de Chronos-2 avec Covariables

On teste ici un fine-tuning complet (100 étapes, strictement sur l'historique d'entraînement avant les 60 derniers jours) avec les 5 covariables macroéconomiques (`DXY_ret`, `VIX`, `TNX_diff`, `OIL_ret`, `SP500_ret`).

> ⚠️ **Temps de calcul :** ~15 minutes sur CPU MacBook Air. Le bloc est encapsulé derrière `RUN_FINETUNING = False` par défaut pour ne pas bloquer l'exécution rapide du notebook.

**Résultats observés lors de l'expérience (100 steps, `lr=1e-5`, `batch_size=32`) :**
- **MAE :** `0.011837` (vs `0.011536` Naïf, `0.011742` Zero-Shot)
- **RMSE :** `0.015968` (vs `0.015653` Naïf, `0.015830` Zero-Shot)
- **Hit Ratio :** `41.67 %` (dégradation due au surapprentissage sur le bruit)
- **Conclusion :** Confirme empiriquement la littérature 2026 : fine-tuner un TSFM généraliste de 120M de paramètres sur une seule série de rendements journaliers bruités ne bat pas le benchmark naïf.

In [ ]:
RUN_FINETUNING = False  # Passer à True pour relancer (~15 min sur CPU)

if RUN_FINETUNING:
    import tempfile, time
    df_train = df.iloc[:-N_TEST].copy()
    train_inputs = [{
        "target": df_train["return"].values,
        "past_covariates": {c: df_train[c].values for c in cov_cols}
    }]
    
    t0 = time.time()
    with tempfile.TemporaryDirectory() as tmpdir:
        ft_pipeline = pipeline.fit(
            inputs=train_inputs,
            prediction_length=1,
            context_length=CTX_LEN,
            finetune_mode="full",
            learning_rate=1e-5,
            num_steps=100,
            batch_size=32,
            output_dir=tmpdir,
            logging_steps=25,
        )
        print(f"Fine-tuning terminé en {time.time() - t0:.1f}s")
        
        wf_ft = ft_pipeline.predict_df(
            batch_multi, prediction_length=1, quantile_levels=[0.1, 0.5, 0.9],
            id_column="item_id", timestamp_column="Date", target="return", freq="B", batch_size=32
        )
        y_pred_ft = wf_ft.sort_values("item_id")["0.5"].values
        print(eval_metrics(res["y_true"], y_pred_ft, "Chronos-2 Fine-Tuné (100 steps)"))